# LeetCode 3515 — Shortest Path in a Weighted Tree bằng Python

- Link: https://leetcode.com/problems/shortest-path-in-a-weighted-tree/
- Độ khó: **Hard**
- Chủ đề: **Euler tour** (cây con ↔ đoạn liên tục), **Fenwick tree cộng đoạn – hỏi điểm**, cập nhật trọng số cạnh = cộng hiệu cho cả cây con

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def treeQueries(self, n: int, edges: List[List[int]], queries: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Hiểu trên cây, "đường đi ngắn nhất từ gốc tới `x`" là **đường duy nhất** ⇒ bằng tổng trọng số các cạnh từ gốc xuống `x`.
2. Nhận ra đổi trọng số cạnh `(cha, con)` một lượng `Δ` làm khoảng cách của **mọi đỉnh trong cây con của `con`** tăng `Δ`.
3. Dùng **Euler tour** (`tin`, `tout`) để cây con thành đoạn `[tin, tout]` trên mảng.
4. Dùng **Fenwick tree** theo kỹ thuật "mảng hiệu": cộng đoạn `O(log n)`, hỏi điểm `O(log n)`.
5. So sánh với cách leo cha và chia khối.

## 2. Đề bài & ví dụ

Cây vô hướng có trọng số, gốc `1`, `n` đỉnh (`1..n`), `edges[i] = [u, v, w]`. Các truy vấn:

- `[1, u, v, w']`: đổi trọng số cạnh `(u, v)` thành `w'` (cạnh chắc chắn tồn tại).
- `[2, x]`: trả về khoảng cách ngắn nhất từ gốc `1` tới `x`.

Trả về đáp án các truy vấn loại 2.

**Ví dụ 1:** `n=2, edges=[[1,2,7]], queries=[[2,2],[1,1,2,4],[2,2]]` → `[7,4]`.

**Ví dụ 2:** `n=3, edges=[[1,2,2],[1,3,4]], queries=[[2,1],[2,3],[1,1,3,7],[2,2],[2,3]]` → `[0,4,2,7]`.

**Ví dụ 3:** `n=4, edges=[[1,2,2],[2,3,1],[3,4,5]], queries=[[2,4],[2,3],[1,2,3,3],[2,2],[2,3]]` → `[8,3,2,5]`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `1 ≤ w, w' ≤ 10^4`, `1 ≤ q ≤ 10^5`.

In [1]:
from typing import List
examples = [
    (2, [[1, 2, 7]], [[2, 2], [1, 1, 2, 4], [2, 2]], [7, 4]),
    (3, [[1, 2, 2], [1, 3, 4]], [[2, 1], [2, 3], [1, 1, 3, 7], [2, 2], [2, 3]], [0, 4, 2, 7]),
    (4, [[1, 2, 2], [2, 3, 1], [3, 4, 5]], [[2, 4], [2, 3], [1, 2, 3, 3], [2, 2], [2, 3]], [8, 3, 2, 5]),
]
for n, e, q, exp in examples:
    print(n, e, q, '-> kỳ vọng', exp)

2 [[1, 2, 7]] [[2, 2], [1, 1, 2, 4], [2, 2]] -> kỳ vọng [7, 4]
3 [[1, 2, 2], [1, 3, 4]] [[2, 1], [2, 3], [1, 1, 3, 7], [2, 2], [2, 3]] -> kỳ vọng [0, 4, 2, 7]
4 [[1, 2, 2], [2, 3, 1], [3, 4, 5]] [[2, 4], [2, 3], [1, 2, 3, 3], [2, 2], [2, 3]] -> kỳ vọng [8, 3, 2, 5]


## 3. Kiến thức nền

### 3.1. Dựng cây có gốc, cha và khoảng cách ban đầu

In [2]:
def rooted(n, edges):
    adj = [[] for _ in range(n + 1)]
    for u, v, w in edges:
        adj[u].append((v, w))
        adj[v].append((u, w))
    par, dist, pw = [0] * (n + 1), [0] * (n + 1), [0] * (n + 1)   # pw[v] = trọng số cạnh (par[v], v)
    order = [1]
    seen = [False] * (n + 1)
    seen[1] = True
    for u in order:
        for v, w in adj[u]:
            if not seen[v]:
                seen[v] = True
                par[v], pw[v], dist[v] = u, w, dist[u] + w
                order.append(v)
    return par, pw, dist, order

print(rooted(4, examples[2][1]))

([0, 0, 1, 2, 3], [0, 0, 2, 1, 5], [0, 0, 2, 3, 8], [1, 2, 3, 4])


### 3.2. Euler tour: cây con là một đoạn

Duyệt DFS (bằng stack), gán `tin[u]` khi vào, `tout[u]` = chỉ số lớn nhất trong cây con. Cây con của `u` = các đỉnh có `tin` trong `[tin[u], tout[u]]`.

In [3]:
def euler(n, par, order):
    children = [[] for _ in range(n + 1)]
    for v in order[1:]:
        children[par[v]].append(v)
    tin, tout = [0] * (n + 1), [0] * (n + 1)
    t = 0
    stack = [(1, False)]
    while stack:
        u, done = stack.pop()
        if done:
            tout[u] = t - 1
            continue
        tin[u] = t; t += 1
        stack.append((u, True))
        for c in reversed(children[u]):
            stack.append((c, False))
    return tin, tout

par, pw, dist, order = rooted(4, examples[2][1])
print(euler(4, par, order))

([0, 0, 1, 2, 3], [0, 3, 3, 3, 3])


### 3.3. Fenwick tree: cộng đoạn – hỏi điểm

Lưu **mảng hiệu** `d`: cộng `Δ` cho `[l, r]` ⇔ `d[l] += Δ`, `d[r+1] −= Δ`. Giá trị tại `i` = tổng tiền tố `d[0..i]`. Fenwick làm cả hai thao tác trong `O(log n)`.

In [4]:
class Fenwick:
    def __init__(self, n):
        self.n = n
        self.t = [0] * (n + 1)

    def add(self, i, v):           # i 0-based
        i += 1
        while i <= self.n:
            self.t[i] += v
            i += i & -i

    def prefix(self, i):           # tổng d[0..i]
        i += 1
        s = 0
        while i > 0:
            s += self.t[i]
            i -= i & -i
        return s

    def range_add(self, l, r, v):
        self.add(l, v)
        if r + 1 < self.n:
            self.add(r + 1, -v)

f = Fenwick(6)
f.range_add(1, 3, 5); f.range_add(2, 5, 2)
print([f.prefix(i) for i in range(6)])

[0, 5, 7, 7, 2, 2]


### 3.4. Cạnh nào là "cạnh cha – con"?

Với truy vấn `[1, u, v, w']`, đỉnh **con** là đỉnh có cha là đỉnh kia: `child = v if par[v] == u else u`. Lượng thay đổi `Δ = w' − (trọng số hiện tại)`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Trên cây, khoảng cách gốc → `x` là tổng các cạnh trên đường đi xuống `x`. Đổi cạnh `(p, c)` (với `c` là con) ảnh hưởng **đúng** những đỉnh mà đường đi từ gốc đi qua cạnh đó — chính là **cây con của `c`**.

Tính tay ví dụ 3 (đường thẳng `1-2-3-4`, trọng số `2, 1, 5`):

| truy vấn | thao tác | khoảng cách |
|---|---|---|
| `[2,4]` | | `2 + 1 + 5 = 8` |
| `[2,3]` | | `3` |
| `[1,2,3,3]` | cạnh (2,3): 1 → 3, Δ = +2, cộng cho cây con của 3 = {3, 4} | |
| `[2,2]` | | `2` (không đổi) |
| `[2,3]` | | `3 + 2 = 5` |

## 5. Cách 1 — Brute Force: BFS lại từ gốc mỗi lần hỏi

In [5]:
from collections import deque

def tree_brute(n, edges, queries):
    w = {}
    adj = [[] for _ in range(n + 1)]
    for u, v, ww in edges:
        adj[u].append(v); adj[v].append(u)
        w[(min(u, v), max(u, v))] = ww
    res = []
    for q in queries:
        if q[0] == 1:
            _, u, v, nw = q
            w[(min(u, v), max(u, v))] = nw
        else:
            x = q[1]
            dist = {1: 0}
            dq = deque([1])
            while dq:
                a = dq.popleft()
                for b in adj[a]:
                    if b not in dist:
                        dist[b] = dist[a] + w[(min(a, b), max(a, b))]
                        dq.append(b)
            res.append(dist[x])
    return res

for n, e, q, exp in examples:
    print(tree_brute(n, e, q), '| kỳ vọng', exp)

[7, 4] | kỳ vọng [7, 4]
[0, 4, 2, 7] | kỳ vọng [0, 4, 2, 7]
[8, 3, 2, 5] | kỳ vọng [8, 3, 2, 5]


**Độ phức tạp:** `O(n)` mỗi truy vấn hỏi ⇒ `O(n q) = 10^{10}`.

**Nút thắt:** một thay đổi cạnh chỉ ảnh hưởng một cây con, và một câu hỏi chỉ cần một đường đi — không cần BFS cả cây.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề:** gọi `c` là đỉnh con của cạnh được cập nhật, `Δ = w' − w_cũ`. Khi đó `dist(y)` tăng `Δ` ⇔ `y` thuộc cây con của `c`.

Với Euler tour, cây con của `c` là đoạn `[tin[c], tout[c]]` ⇒ cập nhật = **cộng đoạn**; truy vấn `dist(x)` = `dist_ban_đầu(x) + (tổng các Δ áp lên vị trí tin[x])` = **hỏi điểm**. Fenwick với mảng hiệu làm cả hai trong `O(log n)`.

Quy tắc:

- **Nếu** cập nhật cạnh `(u, v)` **thì** `c` = đỉnh có cha là đỉnh kia; cộng `Δ` cho `[tin[c], tout[c]]`; lưu trọng số mới.
- **Nếu** hỏi `x` **thì** trả `dist0[x] + prefix(tin[x])`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Leo cha để tính khoảng cách

Lưu `pw[v]` (trọng số cạnh tới cha). Cập nhật `O(1)`; hỏi: cộng `pw` dọc đường từ `x` lên gốc `O(độ sâu)`.

In [6]:
def tree_climb(n, edges, queries):
    par, pw, _, _ = rooted(n, edges)
    res = []
    for q in queries:
        if q[0] == 1:
            _, u, v, nw = q
            c = v if par[v] == u else u
            pw[c] = nw
        else:
            x, s = q[1], 0
            while x != 1:
                s += pw[x]
                x = par[x]
            res.append(s)
    return res

for n, e, q, exp in examples:
    print(tree_climb(n, e, q), '| kỳ vọng', exp)

[7, 4] | kỳ vọng [7, 4]
[0, 4, 2, 7] | kỳ vọng [0, 4, 2, 7]
[8, 3, 2, 5] | kỳ vọng [8, 3, 2, 5]


**Độ phức tạp:** `O(h)` mỗi truy vấn hỏi — xấu nhất `O(n)` (cây đường thẳng).

### 7.2. Cách 3 — Euler tour + Fenwick (lời giải chính)

In [7]:
def tree_fenwick(n, edges, queries):
    par, pw, dist0, order = rooted(n, edges)
    tin, tout = euler(n, par, order)
    fw = Fenwick(n)
    res = []
    for q in queries:
        if q[0] == 1:
            _, u, v, nw = q
            c = v if par[v] == u else u
            delta = nw - pw[c]
            if delta:
                pw[c] = nw
                fw.range_add(tin[c], tout[c], delta)
        else:
            x = q[1]
            res.append(dist0[x] + fw.prefix(tin[x]))
    return res

for n, e, q, exp in examples:
    print(tree_fenwick(n, e, q), '| kỳ vọng', exp)

[7, 4] | kỳ vọng [7, 4]
[0, 4, 2, 7] | kỳ vọng [0, 4, 2, 7]
[8, 3, 2, 5] | kỳ vọng [8, 3, 2, 5]


**Độ phức tạp:** tiền xử lý `O(n)`, mỗi truy vấn `O(log n)`.

### 7.3. Cách 4 — Euler tour + chia khối (cộng đoạn có lazy theo khối)

Thay Fenwick bằng chia khối `√n`: mỗi khối có `lazy`; cộng đoạn = các khối nằm trọn thì cộng `lazy`, phần lẻ cộng trực tiếp; hỏi điểm = `val[i] + lazy[khối]`.

In [8]:
def tree_blocks(n, edges, queries):
    par, pw, dist0, order = rooted(n, edges)
    tin, tout = euler(n, par, order)
    B = max(1, int(n ** 0.5))
    val = [0] * n
    lazy = [0] * (n // B + 1)
    res = []
    for q in queries:
        if q[0] == 1:
            _, u, v, nw = q
            c = v if par[v] == u else u
            d = nw - pw[c]
            pw[c] = nw
            l, r = tin[c], tout[c]
            while l <= r and l % B:
                val[l] += d; l += 1
            while l + B - 1 <= r:
                lazy[l // B] += d; l += B
            while l <= r:
                val[l] += d; l += 1
        else:
            i = tin[q[1]]
            res.append(dist0[q[1]] + val[i] + lazy[i // B])
    return res

for n, e, q, exp in examples:
    print(tree_blocks(n, e, q), '| kỳ vọng', exp)

[7, 4] | kỳ vọng [7, 4]
[0, 4, 2, 7] | kỳ vọng [0, 4, 2, 7]
[8, 3, 2, 5] | kỳ vọng [8, 3, 2, 5]


**Độ phức tạp:** cập nhật `O(√n)`, hỏi `O(1)`.

## 8. TRACE từng bước

Trace cách 3 trên cây 6 đỉnh: `1-2 (3)`, `1-3 (1)`, `2-4 (4)`, `2-5 (2)`, `3-6 (5)`.

In [9]:
def trace(n, edges, queries):
    par, pw, dist0, order = rooted(n, edges)
    tin, tout = euler(n, par, order)
    print('tin  =', {v: tin[v] for v in range(1, n + 1)})
    print('tout =', {v: tout[v] for v in range(1, n + 1)})
    print('dist ban đầu =', {v: dist0[v] for v in range(1, n + 1)})
    fw = Fenwick(n)
    for q in queries:
        if q[0] == 1:
            _, u, v, nw = q
            c = v if par[v] == u else u
            d = nw - pw[c]
            pw[c] = nw
            fw.range_add(tin[c], tout[c], d)
            sub = [y for y in range(1, n + 1) if tin[c] <= tin[y] <= tout[c]]
            print(f"  {q}: cạnh ({par[c]}, {c}) -> Δ = {d:+}, cộng cho đoạn [{tin[c]}, {tout[c]}] = cây con {sub}")
        else:
            x = q[1]
            print(f"  {q}: dist({x}) = {dist0[x]} + {fw.prefix(tin[x])} = {dist0[x] + fw.prefix(tin[x])}")

E6 = [[1, 2, 3], [1, 3, 1], [2, 4, 4], [2, 5, 2], [3, 6, 5]]
Q6 = [[2, 4], [1, 1, 2, 10], [2, 4], [2, 5], [2, 6], [1, 6, 3, 1], [2, 6]]
trace(6, E6, Q6)
print('vét cạn:', tree_brute(6, E6, Q6))

tin  = {1: 0, 2: 1, 3: 4, 4: 2, 5: 3, 6: 5}
tout = {1: 5, 2: 3, 3: 5, 4: 2, 5: 3, 6: 5}
dist ban đầu = {1: 0, 2: 3, 3: 1, 4: 7, 5: 5, 6: 6}
  [2, 4]: dist(4) = 7 + 0 = 7
  [1, 1, 2, 10]: cạnh (1, 2) -> Δ = +7, cộng cho đoạn [1, 3] = cây con [2, 4, 5]
  [2, 4]: dist(4) = 7 + 7 = 14
  [2, 5]: dist(5) = 5 + 7 = 12
  [2, 6]: dist(6) = 6 + 0 = 6
  [1, 6, 3, 1]: cạnh (3, 6) -> Δ = -4, cộng cho đoạn [5, 5] = cây con [6]
  [2, 6]: dist(6) = 6 + -4 = 2
vét cạn: [7, 14, 12, 6, 2]


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def treeQueries(self, n: int, edges: List[List[int]], queries: List[List[int]]) -> List[int]:
        adj = [[] for _ in range(n + 1)]
        for u, v, w in edges:
            adj[u].append((v, w))
            adj[v].append((u, w))
        par, pw, dist0 = [0] * (n + 1), [0] * (n + 1), [0] * (n + 1)
        tin, tout = [0] * (n + 1), [0] * (n + 1)
        seen = [False] * (n + 1)
        seen[1] = True
        t = 0
        stack = [(1, 0)]                       # (đỉnh, chỉ số kề tiếp theo)
        tin[1] = 0; t = 1
        while stack:
            u, i = stack[-1]
            if i < len(adj[u]):
                stack[-1] = (u, i + 1)
                v, w = adj[u][i]
                if not seen[v]:
                    seen[v] = True
                    par[v], pw[v], dist0[v] = u, w, dist0[u] + w
                    tin[v] = t; t += 1
                    stack.append((v, 0))
            else:
                tout[u] = t - 1
                stack.pop()
        bit = [0] * (n + 1)

        def add(i, v):
            i += 1
            while i <= n:
                bit[i] += v
                i += i & -i

        res = []
        for q in queries:
            if q[0] == 1:
                _, u, v, nw = q
                c = v if par[v] == u else u
                d = nw - pw[c]
                if d:
                    pw[c] = nw
                    add(tin[c], d)
                    if tout[c] + 1 < n:
                        add(tout[c] + 1, -d)
            else:
                x = q[1]
                i, s = tin[x] + 1, 0
                while i:
                    s += bit[i]
                    i -= i & -i
                res.append(dist0[x] + s)
        return res

for n, e, q, exp in examples:
    got = Solution().treeQueries(n, e, q)
    print(got, 'OK' if got == exp else 'SAI')

[7, 4] OK
[0, 4, 2, 7] OK
[8, 3, 2, 5] OK


## 10. Vì sao thuật toán đúng

1. **Đường duy nhất:** trên cây, giữa gốc và `x` chỉ có một đường đi đơn ⇒ "ngắn nhất" = tổng trọng số cạnh trên đường đó.
2. **Ảnh hưởng của cập nhật:** cạnh `(p, c)` nằm trên đường gốc → `y` ⇔ `y` là hậu duệ của `c` (kể cả `c`). Vậy thay đổi `Δ` cộng đúng cho cây con của `c`.
3. **Euler tour:** DFS cấp `tin` liên tiếp cho cây con ⇒ cây con của `c` = `{y : tin[c] ≤ tin[y] ≤ tout[c]}`.
4. **Fenwick mảng hiệu:** sau các thao tác cộng đoạn, tổng tiền tố tại `tin[x]` = tổng các `Δ` của mọi đoạn chứa `tin[x]` = tổng thay đổi trên đường gốc → `x`.
5. **Giữ trọng số hiện tại** (`pw[c]`) để tính `Δ` đúng cho các lần cập nhật liên tiếp trên cùng cạnh.

## 11. Phân tích Time / Space Complexity

| Cách | Cập nhật | Hỏi | Space |
|---|---|---|---|
| 1. BFS mỗi lần hỏi | `O(1)` | `O(n)` | `O(n)` |
| 2. Leo cha | `O(1)` | `O(h)` | `O(n)` |
| 3. Euler + Fenwick | `O(log n)` | `O(log n)` | `O(n)` |
| 4. Euler + chia khối | `O(√n)` | `O(1)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'climb': tree_climb, 'fenwick': tree_fenwick, 'blocks': tree_blocks,
         'submit': lambda n, e, q: Solution().treeQueries(n, e, q)}
edge = [
    (1, [], [[2, 1], [2, 1]], [0, 0]),                         # một đỉnh
    (3, [[2, 1, 5], [3, 2, 1]], [[1, 2, 1, 5], [2, 3], [1, 3, 2, 10], [2, 3], [2, 1]], [6, 15, 0]),  # cạnh viết ngược
]
for n, e, q, exp in examples + edge:
    assert tree_brute(n, e, q) == exp
    for name, f in impls.items():
        got = f(n, e, q)
        assert got == exp, (name, n, e, q, got, exp)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với BFS

In [12]:
import random
random.seed(3515)

def rand_tree(n, wmax=20):
    perm = list(range(1, n + 1))
    random.shuffle(perm)
    perm[perm.index(1)], perm[0] = perm[0], 1
    es = []
    for i in range(1, n):
        a, b = perm[random.randrange(i)], perm[i]
        es.append([a, b, random.randint(1, wmax)] if random.random() < 0.5 else [b, a, random.randint(1, wmax)])
    return es

def rand_queries(n, edges, q, wmax=20):
    out = []
    for _ in range(q):
        if edges and random.random() < 0.5:
            u, v, _ = random.choice(edges)
            out.append([1, u, v, random.randint(1, wmax)] if random.random() < 0.5 else [1, v, u, random.randint(1, wmax)])
        else:
            out.append([2, random.randint(1, n)])
    return out

cnt = 0
for _ in range(400):
    n = random.randint(1, 15)
    e = rand_tree(n)
    q = rand_queries(n, e, 20)
    exp = tree_brute(n, e, q)
    for name, f in impls.items():
        assert f(n, e, q) == exp, (name, n, e, q)
    cnt += 1
print('Khớp BFS trên', cnt, 'cây ngẫu nhiên.')

Khớp BFS trên 400 cây ngẫu nhiên.


### 12.3. Ca lớn (`n = q = 10^5`)

In [13]:
import time
random.seed(7)
n = 10**5
for name, e in [('ngẫu nhiên', rand_tree(n, 10**4)), ('đường thẳng', [[i, i + 1, random.randint(1, 10**4)] for i in range(1, n)])]:
    q = rand_queries(n, e, 10**5, 10**4)
    t = time.time(); a = Solution().treeQueries(n, e, q); t1 = time.time() - t
    t = time.time(); b = tree_blocks(n, e, q); t2 = time.time() - t
    assert a == b
    print(f"{name}: Fenwick {t1:.2f}s | chia khối {t2:.2f}s | {len(a)} câu trả lời")

ngẫu nhiên: Fenwick 0.60s | chia khối 0.48s | 50079 câu trả lời


đường thẳng: Fenwick 0.40s | chia khối 2.63s | 50060 câu trả lời


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cộng `w'` thay vì hiệu `Δ = w' − w_cũ`

In [14]:
def wrong_add_new_weight(n, edges, queries):
    par, pw, dist0, order = rooted(n, edges)
    tin, tout = euler(n, par, order)
    fw = Fenwick(n)
    res = []
    for q in queries:
        if q[0] == 1:
            _, u, v, nw = q
            c = v if par[v] == u else u
            fw.range_add(tin[c], tout[c], nw)          # SAI
        else:
            res.append(dist0[q[1]] + fw.prefix(tin[q[1]]))
    return res

n, e, q, exp = examples[0]
print('sai :', wrong_add_new_weight(n, e, q), '| đúng:', exp)

sai : [7, 11] | đúng: [7, 4]


**Vì sao sai:** trọng số cũ vẫn còn trong `dist0`; phải cộng **lượng thay đổi**.

### Lỗi 2 — Không cập nhật trọng số hiện tại sau mỗi lần đổi (Δ tính từ trọng số ban đầu)

In [15]:
def wrong_stale_weight(n, edges, queries):
    par, pw, dist0, order = rooted(n, edges)
    tin, tout = euler(n, par, order)
    fw = Fenwick(n)
    res = []
    for q in queries:
        if q[0] == 1:
            _, u, v, nw = q
            c = v if par[v] == u else u
            fw.range_add(tin[c], tout[c], nw - pw[c])   # SAI: quên pw[c] = nw
        else:
            res.append(dist0[q[1]] + fw.prefix(tin[q[1]]))
    return res

case = (2, [[1, 2, 7]], [[1, 1, 2, 4], [1, 1, 2, 9], [2, 2]])
print('sai :', wrong_stale_weight(*case), '| đúng:', tree_brute(*case))

sai : [6] | đúng: [9]


**Vì sao sai:** lần đổi thứ hai phải tính `Δ` so với trọng số **hiện tại** (4), không phải ban đầu (7).

### Lỗi 3 — Chọn nhầm đỉnh con (luôn lấy `v`)

In [16]:
def wrong_child(n, edges, queries):
    par, pw, dist0, order = rooted(n, edges)
    tin, tout = euler(n, par, order)
    fw = Fenwick(n)
    res = []
    for q in queries:
        if q[0] == 1:
            _, u, v, nw = q
            c = v                                       # SAI
            fw.range_add(tin[c], tout[c], nw - pw[c]); pw[c] = nw
        else:
            res.append(dist0[q[1]] + fw.prefix(tin[q[1]]))
    return res

case = (3, [[1, 2, 2], [2, 3, 1]], [[1, 3, 2, 5], [2, 2], [2, 3]])
print('sai :', wrong_child(*case), '| đúng:', tree_brute(*case))

sai : [5, 6] | đúng: [2, 7]


**Vì sao sai:** truy vấn có thể ghi cạnh theo thứ tự bất kỳ (`[1, 3, 2, …]` với 3 là con của 2).

### Lỗi 4 — DFS đệ quy để tính `tin/tout`

In [17]:
import sys
def euler_recursive(children, u, tin, tout, t):
    tin[u] = t[0]; t[0] += 1
    for c in children[u]:
        euler_recursive(children, c, tin, tout, t)
    tout[u] = t[0] - 1

m = 3000
par_c, _, _, order_c = rooted(m, [[i, i + 1, 1] for i in range(1, m)])
children = [[] for _ in range(m + 1)]
for v in order_c[1:]:
    children[par_c[v]].append(v)
try:
    euler_recursive(children, 1, [0] * (m + 1), [0] * (m + 1), [0])
    print('chạy được')
except RecursionError:
    print('RecursionError trên cây đường thẳng 3000 đỉnh -> dùng stack như lời giải chính')

RecursionError trên cây đường thẳng 3000 đỉnh -> dùng stack như lời giải chính


**Vì sao sai:** độ sâu tới `10^5` vượt giới hạn đệ quy mặc định.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Cập nhật một cạnh/đỉnh làm thay đổi cả **cây con**" ⇒ **Euler tour** để cây con thành đoạn.
- "Cộng đoạn – hỏi điểm" ⇒ Fenwick trên **mảng hiệu** (hoặc cây phân đoạn lazy).
- "Cộng điểm – hỏi tổng cây con" ⇒ Fenwick bình thường trên Euler tour.
- Khoảng cách giữa **hai đỉnh bất kỳ** ⇒ thêm LCA: `d(u) + d(v) − 2 d(lca)`.

## 15. Biến thể của bài

1. Hỏi khoảng cách giữa `u` và `v` bất kỳ ⇒ Fenwick + LCA (cấu trúc cây không đổi).
2. Cập nhật **trọng số đỉnh** thay vì cạnh ⇒ giống hệt (đỉnh `c` ảnh hưởng cây con của `c`).
3. Hỏi **tổng khoảng cách** từ gốc tới mọi đỉnh trong cây con ⇒ Fenwick cộng đoạn – tổng đoạn (hai Fenwick).
4. Bài liên quan: LC 2846, SPOJ QTREE, các bài "Euler tour + BIT".

In [18]:
# Biến thể 1: khoảng cách giữa hai đỉnh bất kỳ (truy vấn [3, u, v])
def build_up(par, n):
    LOG = max(1, n.bit_length())
    up = [par[:]]
    up[0][1] = 1
    for _ in range(1, LOG):
        prev = up[-1]
        up.append([prev[prev[x]] for x in range(n + 1)])
    return up

def two_node_distance(n, edges, queries):
    par, pw, dist0, order = rooted(n, edges)
    tin, tout = euler(n, par, order)
    depth = [0] * (n + 1)
    for v in order[1:]:
        depth[v] = depth[par[v]] + 1
    up = build_up(par, n)
    def lca(a, b):
        if depth[a] < depth[b]: a, b = b, a
        d, j = depth[a] - depth[b], 0
        while d:
            if d & 1: a = up[j][a]
            d >>= 1; j += 1
        if a == b: return a
        for j in range(len(up) - 1, -1, -1):
            if up[j][a] != up[j][b]: a, b = up[j][a], up[j][b]
        return up[0][a]
    fw = Fenwick(n)
    D = lambda x: dist0[x] + fw.prefix(tin[x])
    out = []
    for q in queries:
        if q[0] == 1:
            _, u, v, nw = q
            c = v if par[v] == u else u
            fw.range_add(tin[c], tout[c], nw - pw[c]); pw[c] = nw
        else:
            _, u, v = q
            out.append(D(u) + D(v) - 2 * D(lca(u, v)))
    return out

print(two_node_distance(6, E6, [[3, 4, 6], [1, 1, 2, 10], [3, 4, 6], [3, 4, 5]]))

[13, 20, 6]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Cập nhật / Hỏi | Khi nào dùng |
|---|---|---|---|
| 1 | BFS lại | `O(1)` / `O(n)` | Oracle |
| 2 | Leo cha | `O(1)` / `O(h)` | Cây nông |
| 3 | Euler + Fenwick | `O(log n)` / `O(log n)` | **Nộp bài** |
| 4 | Euler + chia khối | `O(√n)` / `O(1)` | Nhiều truy vấn hỏi |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `treeQueries(3, [[1,2,5],[2,3,5]], [[2,3],[1,2,3,1],[2,3]])`.

**Bài 2 (tính tay):** với cây `E6` ở mục 8, sau `[1, 2, 4, 1]` thì `dist(4)` = ?

**Bài 3 (điền TODO):** viết `subtree_nodes(c)` trả về các đỉnh có `tin` trong `[tin[c], tout[c]]`.

**Bài 4 (tự viết lại):** biến thể 3 — truy vấn `[3, c]`: tổng khoảng cách từ gốc tới mọi đỉnh trong cây con của `c`.

**Bài 5 (mini challenge):** biến thể 2 — mỗi đỉnh có trọng số, đổi trọng số đỉnh, hỏi tổng trọng số đỉnh trên đường gốc → `x`.

In [19]:
def subtree_nodes_todo(tin, tout, c, n):
    # TODO
    raise NotImplementedError

def subtree_distance_sum_todo(n, edges, queries):
    # TODO: hai Fenwick (cộng đoạn – tổng đoạn)
    raise NotImplementedError

def vertex_weights_todo(n, edges, weights, queries):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Ban đầu `dist(3) = 10`. (2) Đổi cạnh (2,3) từ 5 thành 1. (3) `[10, 6]`.

**Bài 2.** (1) `dist0(4) = 3 + 4 = 7`. (2) Δ = `1 − 4 = −3`. (3) `4`.

**Bài 3.** (1) Duyệt mọi đỉnh. (2) So sánh `tin`. (3) Hoặc dùng mảng `who[t]` để lấy đoạn.

**Bài 4.** (1) Tổng = `Σ dist0` cây con (tính sẵn) + tổng các Δ trên đoạn. (2) Cần "cộng đoạn – tổng đoạn": hai Fenwick `B1`, `B2`. (3) Vét cạn kiểm tra.

**Bài 5.** (1) Đổi trọng số đỉnh `c` ⇒ cộng Δ cho cây con của `c`. (2) Ban đầu `dist0` là tổng trọng số đỉnh trên đường. (3) Giống cách 3.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
a1 = Solution().treeQueries(3, [[1, 2, 5], [2, 3, 5]], [[2, 3], [1, 2, 3, 1], [2, 3]])
a2 = Solution().treeQueries(6, E6, [[1, 2, 4, 1], [2, 4]])
assert a1 == [10, 6] and a2 == [4]

def subtree_nodes(tin, tout, c, n):
    return [y for y in range(1, n + 1) if tin[c] <= tin[y] <= tout[c]]

par6, _, _, order6 = rooted(6, E6)
tin6, tout6 = euler(6, par6, order6)
assert sorted(subtree_nodes(tin6, tout6, 2, 6)) == [2, 4, 5]
print('Bài 1:', a1, '| Bài 2:', a2, '| Bài 3:', subtree_nodes(tin6, tout6, 2, 6))

Bài 1: [10, 6] | Bài 2: [4] | Bài 3: [2, 4, 5]


In [21]:
# Bài 4: tổng khoảng cách trong cây con (Fenwick cộng đoạn – tổng đoạn)
class RangeFenwick:
    def __init__(self, n):
        self.b1, self.b2 = Fenwick(n), Fenwick(n)
        self.n = n
    def range_add(self, l, r, v):
        self.b1.add(l, v); self.b2.add(l, v * l)
        if r + 1 < self.n:
            self.b1.add(r + 1, -v); self.b2.add(r + 1, -v * (r + 1))
    def prefix_sum(self, i):            # tổng a[0..i]
        if i < 0: return 0
        return self.b1.prefix(i) * (i + 1) - self.b2.prefix(i)
    def range_sum(self, l, r):
        return self.prefix_sum(r) - self.prefix_sum(l - 1)

def subtree_distance_sum(n, edges, queries):
    par, pw, dist0, order = rooted(n, edges)
    tin, tout = euler(n, par, order)
    base = [0] * n
    for v in range(1, n + 1):
        base[tin[v]] = dist0[v]
    pre = [0]
    for x in base:
        pre.append(pre[-1] + x)
    rf = RangeFenwick(n)
    out = []
    for q in queries:
        if q[0] == 1:
            _, u, v, nw = q
            c = v if par[v] == u else u
            rf.range_add(tin[c], tout[c], nw - pw[c]); pw[c] = nw
        else:
            c = q[1]
            out.append(pre[tout[c] + 1] - pre[tin[c]] + rf.range_sum(tin[c], tout[c]))
    return out

def subtree_distance_brute(n, edges, queries):
    par, pw, _, order = rooted(n, edges)
    tin, tout = euler(n, par, order)
    out = []
    for q in queries:
        if q[0] == 1:
            _, u, v, nw = q
            c = v if par[v] == u else u
            pw[c] = nw
        else:
            c, s = q[1], 0
            for y in subtree_nodes(tin, tout, c, n):
                x = y
                while x != 1:
                    s += pw[x]; x = par[x]
            out.append(s)
    return out

random.seed(8)
for _ in range(200):
    n = random.randint(1, 12)
    e = rand_tree(n)
    qs = [[3 if t[0] == 2 else 1] + t[1:] for t in rand_queries(n, e, 12)]
    qs = [[2, t[1]] if t[0] == 3 else t for t in qs]
    assert subtree_distance_sum(n, e, qs) == subtree_distance_brute(n, e, qs)
print('Bài 4: khớp vét cạn')

Bài 4: khớp vét cạn


In [22]:
# Bài 5: trọng số đỉnh
def vertex_weights(n, edges, weights, queries):
    par, _, _, order = rooted(n, [[u, v, 0] for u, v, *_ in edges])
    tin, tout = euler(n, par, order)
    d0 = [0] * (n + 1)
    for v in order:
        d0[v] = weights[v] + (d0[par[v]] if v != 1 else 0)
    w = weights[:]
    fw = Fenwick(n)
    out = []
    for q in queries:
        if q[0] == 1:
            _, c, nw = q
            fw.range_add(tin[c], tout[c], nw - w[c]); w[c] = nw
        else:
            out.append(d0[q[1]] + fw.prefix(tin[q[1]]))
    return out

def vertex_brute(n, edges, weights, queries):
    par, _, _, _ = rooted(n, [[u, v, 0] for u, v, *_ in edges])
    w, out = weights[:], []
    for q in queries:
        if q[0] == 1:
            w[q[1]] = q[2]
        else:
            x, s = q[1], 0
            while True:
                s += w[x]
                if x == 1: break
                x = par[x]
            out.append(s)
    return out

random.seed(9)
for _ in range(200):
    n = random.randint(1, 12)
    e = rand_tree(n)
    weights = [0] + [random.randint(1, 9) for _ in range(n)]
    qs = [[1, random.randint(1, n), random.randint(1, 9)] if random.random() < 0.5 else [2, random.randint(1, n)] for _ in range(12)]
    assert vertex_weights(n, e, weights, qs) == vertex_brute(n, e, weights, qs)
print('Bài 5: khớp vét cạn')

Bài 5: khớp vét cạn


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
DFS (stack) từ gốc 1: par, pw (trọng số tới cha), dist0, tin, tout
cập nhật (u, v, w'): c = v nếu par[v] == u else u; d = w' - pw[c]; pw[c] = w'
    BIT.add(tin[c], d); BIT.add(tout[c] + 1, -d)
hỏi x: dist0[x] + BIT.prefix(tin[x])
```

**Checklist:**

- [ ] Xác định đúng đỉnh **con** của cạnh?
- [ ] Cộng **Δ**, không cộng trọng số mới?
- [ ] Cập nhật `pw[c]` sau mỗi lần đổi?
- [ ] `tout[c] + 1 < n` trước khi trừ ở mảng hiệu?
- [ ] DFS không đệ quy; trường hợp `n = 1`?